# Notebook 1: Explore the Running Corpus

**Goal**: Understand the training data before building the model.

Before training any ML model, you must know your data deeply.
This notebook answers: What does the corpus look like? What vocabulary does it contain?
How should we set tokenizer hyperparameters?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from 01_data.generate_synthetic import SyntheticRunGenerator

gen = SyntheticRunGenerator(seed=42)
print('Generator ready')

## 1. Activity DataFrame

The structured component of our corpus. Each row will be serialized to a training sentence.

In [ ]:
df = gen.generate_activities(2000)
print(f'Shape: {df.shape}')
df.dtypes

In [ ]:
df.head(8)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

df['run_type'].value_counts().plot(kind='bar', ax=axes[0,0], title='Run types', color='steelblue')
df['distance_miles'].hist(ax=axes[0,1], bins=40, title='Distance (miles)', color='steelblue')
df['pace_min_per_mile'].hist(ax=axes[0,2], bins=40, title='Pace (min/mi)', color='steelblue')
df['avg_heart_rate'].hist(ax=axes[1,0], bins=40, title='Avg heart rate', color='coral')
df['elevation_gain_ft'].hist(ax=axes[1,1], bins=40, title='Elevation gain (ft)', color='coral')
df['perceived_effort'].value_counts().sort_index().plot(kind='bar', ax=axes[1,2], title='Perceived effort', color='coral')

plt.tight_layout()
plt.show()

## 2. Free-text run log notes

This is the **primary training corpus**. The language model will learn to predict the next token in these notes.

In [ ]:
notes = gen.generate_run_notes(5000)

print('Sample run log notes:')
print('=' * 70)
for note in notes[:8]:
    print(note)
    print()

## 3. Corpus statistics

These numbers drive tokenizer and model hyperparameter choices.

In [ ]:
all_words = [w for note in notes for w in note.split()]
unique_words = set(all_words)
total_chars = sum(len(n) for n in notes)

print(f'Documents:          {len(notes):>10,}')
print(f'Total words:        {len(all_words):>10,}')
print(f'Unique words:       {len(unique_words):>10,}')
print(f'Total characters:   {total_chars:>10,}  (~{total_chars/1e6:.1f}M chars)')
print(f'Avg note length:    {total_chars/len(notes):>10.0f} chars')
print(f'Avg note words:     {len(all_words)/len(notes):>10.1f} words')
print()
print(f'Recommended vocab size: 2x-4x unique words = {len(unique_words)*2:,} to {len(unique_words)*4:,}')
print(f'We use 8,000 to balance coverage vs model size.')

In [ ]:
# Word frequency distribution (Zipf's law)
from collections import Counter

word_freq = Counter(all_words)
freqs = sorted(word_freq.values(), reverse=True)

plt.figure(figsize=(10, 4))
plt.loglog(range(1, len(freqs)+1), freqs, 'steelblue', linewidth=1.5)
plt.xlabel('Rank')
plt.ylabel('Frequency')
plt.title("Word frequency distribution (Zipf's law) — running corpus")
plt.grid(True, alpha=0.3)
plt.show()

print('Top 20 most frequent words:')
for word, freq in word_freq.most_common(20):
    print(f'  {word:<20} {freq:>6,}')

## 4. Serialize and preview

This is what the language model actually sees during training.

In [ ]:
from 01_data.preprocessing import RunPreprocessor
from pathlib import Path
import tempfile

# Save sample data to temp dir and serialize
with tempfile.TemporaryDirectory() as tmp:
    raw_dir = Path(tmp) / 'raw'
    raw_dir.mkdir()
    df.to_csv(raw_dir / 'running_activities.csv', index=False)
    with open(raw_dir / 'run_notes.txt', 'w') as f:
        f.write('\n\n'.join(notes[:500]))
    
    prep = RunPreprocessor(raw_dir=raw_dir, output_dir=Path(tmp) / 'processed')
    corpus = prep.build_corpus()

print(f'Total corpus documents: {len(corpus):,}')
print()
print('First 3 documents (model training inputs):')
for doc in corpus[:3]:
    print(' ', doc)
    print()

## Exercise

1. Find the 5 least common words. Would they get their own BPE tokens, or be split into subwords?
2. Modify `generate_run_notes()` to add entries about injuries or weather. Does the word distribution shift?
3. What is the min/max note length? Would very short notes (< 5 words) be useful training data?